# 🀄 Train the on-phone mahjong tile detector (free)

Run this in **Google Colab** (free): *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

It downloads the free **Mahjong Tiles Universal** dataset (Roboflow Universe, CC BY 4.0, 42 tile classes incl. 8 flowers/seasons), trains a small YOLO model (~15–30 min) and gives you two files:

* `tiles.onnx` – the model (~10 MB)
* `tiles.json` – the tile names

Put both in the `models/` folder of the app repo and `git push`, then open **More → 📷 Tile detector (beta) → Load from website**.

You need a free Roboflow account for the API key: https://app.roboflow.com → Settings → API Keys.

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
ROBOFLOW_API_KEY = "paste-your-key-here"  # free key from app.roboflow.com → Settings → API Keys

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("test-ag8z6").project("mahjong-tiles-universal")
version = project.versions()[0]  # newest version
ds = version.download("yolov8")
print(ds.location)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # small & fast for phones
model.train(data=f"{ds.location}/data.yaml", epochs=80, imgsz=640, batch=16, patience=20)
metrics = model.val()
print('mAP50:', metrics.box.map50)

In [ ]:
import json, shutil
best = YOLO(model.trainer.best)
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(onnx_path, "tiles.onnx")
names = [best.names[i] for i in range(len(best.names))]
json.dump({"names": names, "imgsz": 640, "map": {}, "source": "Mahjong Tiles Universal (Roboflow Universe, CC BY 4.0)"}, open("tiles.json", "w"), indent=1)
print(names)

## Check the tile names

The app understands names like `1m`, `5p`, `9s`, `1z`–`7z` (East, South, West, North, White, Green, Red) and `f1`–`f8` (assumed flowers 1–4, seasons 1–4). If a name is different, add it to `"map"` in `tiles.json`, e.g. `"map": {"f5": "S1"}`. Codes: `m1`–`m9`, `p1`–`p9`, `s1`–`s9`, `w1`–`w4`, `d1` red, `d2` green, `d3` white, `F1`–`F4` flowers, `S1`–`S4` seasons, `A:cat` …

In [ ]:
from google.colab import files
files.download("tiles.onnx")
files.download("tiles.json")